# Medium-Effort Optimization Playbook

The **six MEDIUM-effort levers** — architectural moves on well-trodden paths, a step up from the one-line tweaks of the LOW tier.

| # | Lever | What it does |
|---|---|---|
| 1 | LLM routing | Classify each request, send it to the cheapest model that can handle it |
| 2 | Bedrock Guardrails | Block / redact at the API layer — refused requests never cost generation |
| 3 | RAG / indexing | Retrieve only the top-K chunks the query needs, not the whole corpus |
| 4 | Prompt compression | Strip low-signal tokens from the dynamic context before the model sees it |
| 5 | Conversation & memory | Bound per-turn context (sliding window + summary), persist facts across sessions |
| 6 | Batch inference | Run offline work asynchronously at 50% of the on-demand price |

Each lever is independent — lift any one into your own application without the others. Run the setup cell, then work through them in order.

## Setup

In [ ]:
import boto3, time, json
REGION = "us-east-1"
RUNTIME = boto3.client("bedrock-runtime", region_name=REGION)

HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET = "global.anthropic.claude-sonnet-4-6"

## 1. LLM Routing — Lever 07

Query complexity has a long tail. In a typical workload only ~20% of requests genuinely need a workhorse like Sonnet 4.6 or Opus 4.8; the other ~80% are factual lookups or single-step classifications Haiku 4.5 answers just as well at ~**1/3 the cost**. Routing classifies each request, then sends it to the *cheapest* model that can handle it.

There's a spectrum of how you classify, trading setup effort against routing quality:

| Pattern | How it works | Latency | Quality |
|---|---|---|---|
| Static rules | Token count / language / code-fence → bucket | Sub-ms | Brittle but transparent |
| **LLM classifier** | Cheap LLM emits a complexity/intent label | ~100-300ms | Good if the prompt is good |
| Cascade (FrugalGPT) | Cheap model first; escalate on low confidence | Variable | Best case = small-model only |
| Preference classifier (RouteLLM) | Trained on human preference data | Sub-ms | ~95% strong-model quality at ~26% cost |
| Generative router (Arch-Router) | Small LM emits a route token | ~50-150ms | SOTA on multi-turn agents |

We build the **LLM-classifier** pattern below — the practical starting point. Two rules keep it a win:

- **Classifier must be tiny and deterministic** — `temperature=0`, `maxTokens≈5`, and **cache its static prompt** (Lever 04).
- **The cheap path must take the majority of traffic** — and you must confirm routed-down quality on your own eval set, not the vendor's benchmark.

The failure mode to watch is the floor: if the classifier round-trip makes the "simple" path slower or pricier than just calling the big model, routing has cost you.

In [ ]:
def classify_complexity(query: str) -> str:
    resp = RUNTIME.converse(
        modelId=HAIKU,
        messages=[{"role": "user", "content": [{"text": (
            "Classify this query as one word: 'simple' (factual lookup, single-step) "
            "or 'complex' (multi-step reasoning, ambiguous, policy interpretation).\n\n"
            f"Query: {query}\n\nClassification:"
        )}]}],
        inferenceConfig={"maxTokens": 5, "temperature": 0},
    )
    return resp["output"]["message"]["content"][0]["text"].strip().lower()

def route_and_call(query: str):
    label = classify_complexity(query)
    target = HAIKU if "simple" in label else SONNET
    resp = RUNTIME.converse(
        modelId=target,
        messages=[{"role": "user", "content": [{"text": query}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0},
    )
    return {
        "label": label,
        "routed_to": target.split(".")[-1][:25],
        "text": resp["output"]["message"]["content"][0]["text"],
        "output_tokens": resp["usage"]["outputTokens"],
    }

test_queries = [
    "Where is my order #12345?",                                            # simple
    "What's your return policy on opened headphones?",                       # simple
    "My laptop arrived damaged but it's been 35 days; what can you do?",     # complex (policy edge case)
    "I bought 3 phones, returned 2 last week, want to return the third now "
    "but the original receipt says they were a bundle deal — refund amount?",  # complex
]

for q in test_queries:
    r = route_and_call(q)
    print(f"\n[{r['label']:>8}] -> {r['routed_to']}  ({r['output_tokens']} out)")
    print(f"  Q: {q[:80]}")
    print(f"  A: {r['text']}")

**What you'll see**: ~80% of typical queries route to Haiku → ~3× cheaper for those, while the classifier adds only ~5 output tokens. Watch the `[label] -> routed_to` column: each simple lookup lands on Haiku, each policy edge-case escalates to Sonnet.

---

## 2. Bedrock Guardrails — Lever 08

Guardrails are sold as **safety** — block toxic prompts, refuse off-scope topics, redact PII. They're equally a **cost-and-latency** lever: a request blocked at the input layer never reaches the model, so you don't pay generation tokens on injection probes, jailbreaks, or off-topic traffic. On any public endpoint, abuse traffic is constant, and an off-topic "write me a poem" costs the same tokens as a real question.

A guardrail bundles **independent** policies, evaluated separately, applied differently to **input** vs **output**:

| Policy | Catches | Action |
|---|---|---|
| Content filters | Hate, insults, sexual, violence, misconduct (text + images) | Strength LOW/MED/HIGH per category |
| Prompt-attack detection | Jailbreak / prompt-injection (input-side) | Block |
| Denied topics | Off-scope subjects defined in natural language | Block |
| Word filters | Custom block lists + managed profanity | Block |
| Sensitive info (PII) | Email, phone, SSN, cards, names + your regex | **BLOCK** or **ANONYMIZE** |
| Contextual grounding | RAG hallucinations — scores grounding + relevance | Block below threshold |

We demo two ways to call it: **inline** with `converse` (one guardrailed call), then the **standalone `ApplyGuardrail` API** — the cost trick that screens text *without invoking a model*, stopping bad traffic before it costs you retrieval **and** generation.

<div class="alert alert-block alert-info">

**Pricing is per "text unit" ≈ 1,000 characters**, billed per policy evaluated — so blocking 5% off-topic traffic at the input layer saves ~5% of your *full* request cost (retrieval + generation), which more than funds the eval fee. Batch screening in 1,000-char multiples to avoid waste.

</div>

In [ ]:
BEDROCK = boto3.client("bedrock", region_name=REGION)

# Create a minimal guardrail (ephemeral — cleaned up at end of this section)
guardrail = BEDROCK.create_guardrail(
    name=f"playbook-workshop-{int(time.time())}",
    description="Workshop demo — blocks competitor questions and redacts PII",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Competitor questions",
                "definition": "Questions comparing us to or recommending named competitors",
                "examples": ["Why is the other store cheaper?", "Should I shop somewhere else instead?"],
                "type": "DENY",
            }
        ]
    },
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "EMAIL", "action": "ANONYMIZE"},
            {"type": "PHONE", "action": "ANONYMIZE"},
            {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
        ]
    },
    blockedInputMessaging="Sorry, I can't answer questions about competitors.",
    blockedOutputsMessaging="Output blocked by safety policy.",
)
GUARDRAIL_ID = guardrail["guardrailId"]
GUARDRAIL_VERSION = "DRAFT"
print(f"Created guardrail: {GUARDRAIL_ID}")

In [ ]:
def converse_with_guardrail(query):
    return RUNTIME.converse(
        modelId=SONNET,
        messages=[{"role": "user", "content": [{"text": query}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0},
        guardrailConfig={
            "guardrailIdentifier": GUARDRAIL_ID,
            "guardrailVersion": GUARDRAIL_VERSION,
            "trace": "enabled",
        },
    )

for q in [
    "How does your warranty compare to the store down the street?",  # should be blocked (competitor)
    "My email is alice@example.com — can you confirm receipt?",      # PII anonymized
    "What's your warranty on a refurbished laptop?",                  # should pass
]:
    r = converse_with_guardrail(q)
    stop = r.get("stopReason", "end_turn")
    text = r["output"]["message"]["content"][0].get("text", "<no text>")
    print(f"\n[{stop:>20}] {q}\n  -> {text}")

**The cost trick — `ApplyGuardrail` standalone.** The inline call above still ran the model on the *allowed* requests. The bigger win is screening the **input before any model or retrieval call** with the standalone API: bad traffic is refused for the price of one guardrail eval, not a full inference (and it works against non-Bedrock models too — one guardrail, anywhere in your pipeline).

In [ ]:
# Standalone ApplyGuardrail: screen INPUT with NO model invocation.
for label, text in [
    ("competitor", "Should I just shop at the store down the street instead?"),  # denied topic
    ("legitimate", "What's your warranty on a refurbished laptop?"),             # passes
]:
    r = RUNTIME.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source="INPUT",
        content=[{"text": {"text": text}}],
    )
    print(f"[{label:11}] action={r['action']:22} (no model called)")
    if r["action"] == "GUARDRAIL_INTERVENED":
        print("              -> refused for the price of one guardrail eval, not a full inference")

In [ ]:
BEDROCK.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
print(f"Deleted guardrail {GUARDRAIL_ID}")

**What you'll see**: the competitor question returns `stopReason=guardrail_intervened` (inline) and `GUARDRAIL_INTERVENED` (standalone) — no real answer generated, so it bills a fraction of a full inference. The PII email is anonymized in place; legitimate queries pass. At a 5% probe rate, those early refusals are real savings.

---

## 3. RAG / Indexing — Lever 09

The instinct on a large knowledge source is to stuff it all into context. That's expensive *and* counterproductive: past a certain size accuracy drops because the model underweights information buried mid-prompt (the "lost in the middle" effect). RAG inverts it — embed the corpus once, then per query retrieve only the top-K relevant chunks. The cost win is the retrieve step: 3-5K tokens of relevant context instead of a 50K-token dump.

Four levers carry most of the win, and we run them against a **dedicated playbook Knowledge Base** — the workshop stack builds it, uploads docs with per-document metadata, and **ingests it for you** (separate from the Developer-Journey KB), so it's queryable the moment you arrive:

| Lever | What it does | We demo |
|---|---|---|
| **Chunking strategy** | How docs are split — dominates retrieval quality | Read back the KB's config (set on the data source) |
| **top-K tuning** | How many chunks you pass | `numberOfResults` 10 → 5 → 3 |
| **Metadata filtering** | Filter at query time (product line, doc type) | `filter` on `retrieve` |
| **Reranking** | Cross-encoder reorders candidates, best chunk first | `rerankingConfiguration` on `retrieve` |

<div class="alert alert-block alert-warning">
<b>Cache interaction:</b> retrieved chunks are <b>dynamic</b> — keep them <i>after</i> your <code>cachePoint</code>. Putting them before the cached prefix busts the cache on every query (Lever 04).
</div>

In [ ]:
# Read the playbook KB id + data-source id from SSM. The workshop stack already
# created the KB, uploaded docs, and RAN INGESTION at deploy time — so it's queryable now.
sts = boto3.client("sts", region_name=REGION)
account_id = sts.get_caller_identity()["Account"]
ssm = boto3.client("ssm", region_name=REGION)

def _param(name):
    return ssm.get_parameter(Name=name)["Parameter"]["Value"]

KB_ID = DS_ID = ""
try:
    KB_ID = _param(f"/{account_id}-{REGION}/playbook-kb/knowledge-base-id")
    DS_ID = _param(f"/{account_id}-{REGION}/playbook-kb/data-source-id")
    print("Playbook KB:", KB_ID, "| data source:", DS_ID)
except Exception as e:
    print("Could not read the playbook KB params from SSM:", e)
    print("Deploy the workshop stack — it creates /{account}-{region}/playbook-kb/*.")

AGENT_RT = boto3.client("bedrock-agent-runtime", region_name=REGION)
AGENT = boto3.client("bedrock-agent", region_name=REGION)
QUERY = "What's the warranty on a refurbished laptop?"

# Inspect the chunking strategy the KB was built with — it lives on the data source.
if KB_ID and DS_ID:
    ds = AGENT.get_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
    chunk = ds["dataSource"]["vectorIngestionConfiguration"]["chunkingConfiguration"]
    print("\nChunking strategy:", chunk["chunkingStrategy"])
    print(json.dumps(chunk, indent=2, default=str))
else:
    print("Skipped — no playbook KB params available.")

**What you'll see**: the data source reports `HIERARCHICAL` chunking — parent chunks (~1500 tokens) for context, child chunks (~300 tokens) that are actually searched. This is set on the data source and **applied by the ingestion job**; the two are coupled, so to change chunking you'd update the config and re-ingest.

<div class="alert alert-block alert-info">

**Why hierarchical is the default.** You search the small child chunks (precise vector matches) but hand the model the larger parent (surrounding context) — best of both. Alternatives: <b>semantic</b> (split on embedding-similarity breakpoints — better for fuzzy document boundaries, more compute) and <b>fixed-size</b> (uniform windows — cheap baseline). Start hierarchical; A/B against semantic on your eval set.

</div>

In [ ]:
# --- Lever: top-K tuning ---
# The simplest cost lever and the most over-set. Tune K *down* and watch the context shrink.
if KB_ID:
    for top_k in [10, 5, 3]:
        resp = AGENT_RT.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": QUERY},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": top_k}},
        )
        chunks = resp["retrievalResults"]
        chars = sum(len(c["content"]["text"]) for c in chunks)
        print(f"top_k={top_k:2}  chunks={len(chunks):2}  total_chars={chars:5}")
else:
    print("Skipped — no playbook KB available.")

In [ ]:
# --- Lever: metadata filtering ---
# Each doc was uploaded with a .metadata.json companion ({product_line, doc_type}).
# Filtering at query time shrinks the candidate set BEFORE the vector search —
# better precision, multi-tenant isolation, and less to rank.
if KB_ID:
    # Only policy docs, only the 'laptops' or 'all' product lines.
    flt = {"andAll": [
        {"equals": {"key": "doc_type", "value": "policy"}},
        {"in": {"key": "product_line", "value": ["laptops", "all"]}},
    ]}
    for label, cfg in [("no filter", {}), ("filtered", {"filter": flt})]:
        resp = AGENT_RT.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": QUERY},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 5, **cfg}},
        )
        hits = resp["retrievalResults"]
        print(f"{label:10}: {len(hits)} chunks", end="")
        if hits:
            md = hits[0].get("metadata", {})
            print(f"  | top chunk meta: doc_type={md.get('doc_type')}, product_line={md.get('product_line')}")
        else:
            print()
else:
    print("Skipped — no playbook KB available.")

In [ ]:
# --- Lever: reranking ---
# A second-stage cross-encoder reorders the retrieved candidates by true relevance
# and puts the best chunk first — countering "lost in the middle". It usually lets
# you pass FEWER chunks at higher accuracy, so reach for it before raising K.
RERANK_MODEL = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.rerank-v1:0"

if KB_ID:
    try:
        resp = AGENT_RT.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": QUERY},
            retrievalConfiguration={"vectorSearchConfiguration": {
                "numberOfResults": 10,   # retrieve a wider candidate set...
                "rerankingConfiguration": {
                    "type": "BEDROCK_RERANKING_MODEL",
                    "bedrockRerankingConfiguration": {
                        "numberOfRerankedResults": 3,   # ...then keep the best 3
                        "modelConfiguration": {"modelArn": RERANK_MODEL},
                    },
                },
            }},
        )
        print(f"reranked to {len(resp['retrievalResults'])} chunks (best-first):")
        for i, ch in enumerate(resp["retrievalResults"], 1):
            print(f"  {i}. score={ch.get('score', 0):.3f}  {ch['content']['text'][:70]}...")
    except Exception as e:
        print("Reranking not available in this account/region:", str(e)[:120])
        print("(Amazon Rerank must be enabled in Bedrock model access — the API shape above is correct.)")
else:
    print("Skipped — no playbook KB available.")

**What you'll see**: 
- **top-K** 10→3 shrinks retrieved context by roughly half with no quality loss on a well-scoped query.
- **Metadata filtering** narrows to just the policy/laptop docs *before* the vector search — fewer, more precise candidates.
- **Reranking** retrieves a wide set then keeps the best 3 by cross-encoder score (best-first), which is why you reach for it before raising K.

The production order of operations: **filter → retrieve → rerank → trim K** — each step makes the context smaller and more relevant. Confirm the trade-offs on YOUR eval set.

---

## 4. Prompt Compression — Lever 10

Compression treats your prompt as a token stream with uneven information density. A small "compression model" — **LLMLingua-2**, a BERT-class token classifier — scores each token's value and drops the low-signal ones. The main model sees a shorter prompt, returns the same answer, you pay less.

| Approach | Compresses to | Accuracy |
|---|---|---|
| Original prompt | 100% | baseline |
| Selective Context (2023) | ~60% | ~baseline |
| LLMLingua (2023) | ~33% | slight drop |
| **LLMLingua-2 (2024)** | **~22%** | **↑ on some tasks** |

LLMLingua-2 is the frontier — task-agnostic, ~1/4 the tokens, and on some benchmarks the *compressed* prompt beats the original (removed tokens were noise). The win scales with how much dynamic context you push: at high RAG volume, dropping chunks to ~1/3 is a direct ~3× cut on that slice of the input bill.

<div class="alert alert-block alert-warning">
<b>The cache gotcha — the load-bearing rule.</b> Compression rewrites the prompt, and cache keys match the longest token prefix from position 0. A rewritten prefix is a permanent cache miss. So the decision order is: <b>cache the stable prefix → use context editing for tool-heavy loops → compress only the remaining large, dynamic, un-cacheable tail</b> (RAG chunks, user docs). Never compress a cached system prompt.
</div>

<div class="alert alert-block alert-info">
<b>LLMLingua-2 runs in-process — no endpoint required.</b> <code>pip install llmlingua</code> pulls in <code>transformers</code> + <code>torch</code>; <code>PromptCompressor(...)</code> downloads the BERT-class model (~500 MB on first use) straight into the kernel and compresses locally — no SageMaker, no API call. It's a <b>demo</b> here only because of that one-time model download; the cell runs the real library if it's installed and otherwise prints the principle and skips cleanly. (In <i>production</i> you'd often host it on a GPU SageMaker/ECS endpoint so the compression latency doesn't block your app — an optional scaling choice, not a prerequisite.)
</div>

In [ ]:
# LLMLingua-2 on a RAG chunk. Compress ONLY this dynamic content — never a cached prefix.
RAG_CHUNK = (
    "The 30-day return policy applies to all undamaged products that include their original packaging. "
    "Customers may initiate a return online by visiting the support portal and entering their order number. "
    "Once the return is approved, a prepaid shipping label will be emailed within one business day. "
    "Refunds are processed within 5-7 business days after the returned item is received and inspected. "
    "Items damaged in transit are replaced free of charge once photos of the damage are submitted."
)

try:
    from llmlingua import PromptCompressor

    # Loaded once at startup and reused across requests (BERT-class, small + fast).
    compressor = PromptCompressor(
        model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
        use_llmlingua2=True,
    )
    result = compressor.compress_prompt(
        RAG_CHUNK,
        rate=0.33,                            # keep ~1/3 of tokens
        force_tokens=["\n", ".", "?", "!"],   # never drop these structural tokens
    )
    print(f"origin tokens:     {result['origin_tokens']}")
    print(f"compressed tokens: {result['compressed_tokens']}  ({result['rate']})")
    print(f"\ncompressed prompt:\n{result['compressed_prompt']}")
    print("\n-> feed result['compressed_prompt'] as the volatile tail, AFTER your cachePoint.")
except ImportError:
    print("llmlingua not installed — `pip install llmlingua` to run this live.")
    print("Principle: LLMLingua-2 keeps high-signal tokens and drops filler,")
    print("compressing RAG context to ~1/3 its size for the same answer — applied only")
    print("to the dynamic tail, never to a cached system prompt.")

**What you'll see**: the compressed prompt keeps the high-signal tokens and drops filler — same answer, roughly a third of the tokens. The rule that matters either way: compress only dynamic content (RAG chunks, user docs), never a cached prefix. For real workloads: [LLMLingua repo](https://github.com/microsoft/LLMLingua).

---

## 5. Conversation & Memory — Lever 11

A multi-turn agent leaks context in two directions, and they're the same discipline at two timescales: **decide what enters the context window, and when.**

- **Within a session** — the naive approach replays the whole history every turn. By turn 50 each request carries 49 prior turns, so per-turn cost climbs linearly until you hit the context window and the model errors out.
- **Across sessions** — the agent starts cold tomorrow and re-asks for preferences it already learned.

The fix within a session is **sliding window + rolling summary**: keep the last N turns verbatim, fold older turns into a periodic summary, so per-turn cost stays roughly constant. The knobs:

| Knob | Typical | Tune by |
|---|---|---|
| Window size N | 5-10 turns | Model loses recent context → raise N; cost creeps → lower N |
| Summary trigger | Every M turns, or a token threshold | Threshold-based is more robust than turn-count |
| Summary model | A cheaper model than the main one | Small/fast for the summary, the workhorse for the reply |

Across sessions, **managed memory** (AgentCore) persists what matters: raw turns go to short-term storage, and an async pass *extracts* durable facts into a namespaced store you read back later — so you retrieve a handful of relevant facts instead of resending the transcript. (Anthropic also ships managed helpers for the within-session half — **context editing** auto-clears stale tool results; the **memory tool** is a file-backed store — often a better first reach than hand-rolling.)

<div class="alert alert-block alert-warning">
<b>The cache interaction.</b> Trimming history modifies the prompt prefix, invalidating downstream cache breakpoints (Lever 04). Cache up to the stable <i>summary</i>, leave only the recent N turns dynamic, and tune the trigger so you're not re-trimming every turn and thrashing the cache.
</div>

We first measure full-history vs sliding-window growth, run a real rolling-summary manager, then move to AgentCore Memory for the cross-session half.

In [ ]:
# First, SEE the problem: naive full-history vs a sliding window, as a conversation grows.
# A realistic 24-turn support conversation (a multi-issue session).
CONVO_TURNS = [
    ("Hi, the monitor from order #A-12345 has a dead pixel.", "Sorry about that — when did it arrive?"),
    ("About three weeks ago.", "You're within the 30-day window: full refund or replacement?"),
    ("Replacement, same model.", "Done. Is the shipping address on file still correct?"),
    ("Yes, same address.", "Great. I'll queue the replacement monitor."),
    ("Also, my keyboard from order #B-67890 has a sticking spacebar.", "Noted — is it within warranty?"),
    ("Bought it two months ago.", "That's covered by the 1-year manufacturer warranty."),
    ("Can you ship both together?", "I can bundle the replacement monitor and a warranty keyboard."),
    ("What's the ETA?", "Typically 2-3 business days once processed."),
    ("Do I need to return the old monitor?", "Yes — a prepaid label will be emailed to you."),
    ("And the keyboard?", "Keep using it until the replacement arrives, then return it with the same label."),
    ("Will I get one tracking number or two?", "One combined shipment, one tracking number."),
    ("Perfect, thanks.", "Anything else I can help with today?"),
]

def to_messages(turns):
    msgs = []
    for u, a in turns:
        msgs.append({"role": "user", "content": [{"text": u}]})
        msgs.append({"role": "assistant", "content": [{"text": a}]})
    return msgs

def est_tokens(messages):
    return sum(len(c["text"]) for m in messages for c in m["content"]) // 4  # rough

print(f"{'turn':>4}  {'full_history':>13}  {'sliding(4)':>11}")
for t in [1, 4, 8, 12]:
    so_far = CONVO_TURNS[:t]
    full = est_tokens(to_messages(so_far))
    slide = est_tokens(to_messages(so_far[-4:]))
    print(f"{t:>4}  {full:>13}  {slide:>11}")
print("\n-> full history grows every turn; a window stays flat — but a bare window forgets.")

**What you'll see**: full history grows linearly while the sliding window stays bounded. But a bare window *forgets* — drop turn 3 and the agent loses the order number. The production pattern adds a **rolling summary**: fold dropped turns into a running summary so nothing important is lost while per-turn cost stays roughly constant.

You don't hand-roll this. The **Strands Agents SDK** ships it natively — `SummarizingConversationManager` (and `SlidingWindowConversationManager`) plug into an `Agent` via `conversation_manager=`. This is **purely client-side context management** — it is *not* AgentCore Memory, needs no AWS resource or role, and runs on the same Bedrock model calls. The cell below wires it to a real agent.

In [ ]:
# Strands-native within-session memory — client-side, NOT AgentCore (no AWS resource/role).
from strands import Agent
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager

# A cheap model summarizes the dropped turns; the lead answers on its own model.
summarizer = Agent(
    model=BedrockModel(model_id=HAIKU, region_name=REGION),
    system_prompt="Summarize the conversation so far in <=3 bullets: facts, decisions, open questions.",
)

# preserve_recent_messages = the verbatim window; summary_ratio = how much of the
# overflow to fold into the running summary when the window is exceeded.
mgr = SummarizingConversationManager(
    summary_ratio=0.5,
    preserve_recent_messages=6,
    summarization_agent=summarizer,
)

agent = Agent(
    model=BedrockModel(model_id=HAIKU, region_name=REGION),
    conversation_manager=mgr,
    system_prompt="You are a concise customer-support agent.",
)

# Drive a realistic multi-turn support conversation.
convo = [
    "Hi — my monitor from order #A-12345 has a dead pixel.",
    "It arrived about three weeks ago.",
    "I'd like a replacement, same model.",
    "My shipping address on file is fine.",
    "Separately, the spacebar on my keyboard order #B-67890 is sticking.",
    "Can you ship the replacement monitor and a fix for the keyboard together?",
    "What's the ETA on the combined shipment?",
    "Will I get a single tracking number or two?",
]
for turn in convo:
    agent(turn)

print(f"messages retained after {len(convo)} turns: {len(agent.messages)}")
print(f"manager: {type(agent.conversation_manager).__name__}")

# The order numbers from turns 1 and 5 should survive into the bounded context.
recall = agent("Remind me which two order numbers we've been discussing.")
print("\nrecall:", recall.message["content"][0]["text"])

**What you'll see**: the manager keeps the recent window verbatim and folds older turns into a summary, so the message count stays bounded as the conversation grows — yet the agent still recalls **both** order numbers (#A-12345 from turn 1, #B-67890 from turn 5) because the summary preserved them. Bounded per-turn cost *with* memory of what mattered, in a few lines of SDK config.

That bounds cost *within* a session. The other half of the lever — remembering Alex across *sessions* — is **AgentCore Memory**, next.

### Cross-session memory — AgentCore Memory

The sliding window keeps **one session** bounded. The next problem is *across* sessions: when Alex returns tomorrow, you don't want to replay the whole transcript to re-learn that they're on the Pro plan. **AgentCore Memory** is the managed service for this — no infrastructure to run:

- **Short-term** — raw turns of a session, recalled verbatim with `get_last_k_turns`.
- **Long-term** — facts and preferences asynchronously *extracted* from those turns into a searchable vector store, recalled later (in any future session).

You pick which extraction strategies to enable; AgentCore manages the prompt, model, and schema, landing records under a namespace template:

| Strategy | Extracts | Default namespace |
|---|---|---|
| **Semantic** | Standalone facts about the user/world | `/users/{actorId}/facts/` |
| **User preference** | Stable per-user preferences | `/users/{actorId}/preferences/` |
| **Summary** | Rolling conversation summary | `/sessions/{sessionId}/summary/` |
| **Episodic** | Meaningful interaction sequences | `/episodes/{actorId}/` |

We'll do the full lifecycle: **create** a memory resource (you need to know how), write a realistic support conversation as events, recall it short-term, then — in a fresh session — let a **Strands agent** retrieve the extracted facts through the SDK-native `AgentCoreMemoryToolProvider`, the documented bridge between Strands and AgentCore Memory.

<div class="alert alert-block alert-warning">
<b>Creating a memory needs an execution role.</b> AgentCore assumes a role (trust: <code>bedrock-agentcore.amazonaws.com</code> + <code>bedrock:InvokeModel</code>) to run the async extraction. The workshop stack provisions one and stores its ARN in SSM — the cell reads it from there. Namespaces drive both retrieval and IAM, are <b>hard to migrate later</b>, and extraction is <b>asynchronous and not free</b> (each pass calls a Bedrock model in your account).
</div>

In [ ]:
import os, time
from bedrock_agentcore.memory import MemoryClient

# AgentCore Memory needs an execution role it assumes to call Bedrock for the
# async fact-extraction pass (trust: bedrock-agentcore.amazonaws.com + bedrock:InvokeModel).
# The workshop stack provisions one and stores its ARN in SSM; we read it from there,
# falling back to the MEMORY_EXECUTION_ROLE_ARN env var.
MEMORY_ROLE_ARN = os.environ.get("MEMORY_EXECUTION_ROLE_ARN", "")
if not MEMORY_ROLE_ARN:
    try:
        ssm = boto3.client("ssm", region_name=REGION)
        MEMORY_ROLE_ARN = ssm.get_parameter(
            Name="/app/customersupport/agentcore/runtime_iam_role"
        )["Parameter"]["Value"]
    except Exception as e:
        print("Could not read the execution-role ARN from SSM:", e)

MEMORY_READY = False
memory_id = None
ACTOR_ID = "alex-99"                        # stable per end-user; templates the namespace
SESSION_ID = f"sess-{int(time.time())}"     # this conversation

if MEMORY_ROLE_ARN:
    mem = MemoryClient(region_name=REGION)

    # Create a memory resource WITH a long-term strategy. The semantic strategy
    # extracts durable facts from each session into the {actorId}-templated namespace.
    memory = mem.create_memory_and_wait(
        name="PlaybookMemory",
        description="Conversation memory for Lever 11 hands-on",
        strategies=[
            {
                "semanticMemoryStrategy": {
                    "name": "UserFacts",
                    "namespaces": ["/users/{actorId}/facts"],
                }
            }
        ],
        event_expiry_days=7,
        memory_execution_role_arn=MEMORY_ROLE_ARN,
    )
    memory_id = memory["id"]
    MEMORY_READY = True
    print("Memory:", memory_id, memory["status"])
else:
    print("No execution-role ARN found — skipping the AgentCore Memory cells.")
    print("Deploy the workshop stack, or export MEMORY_EXECUTION_ROLE_ARN, to run this section.")

In [ ]:
# Write a realistic support conversation as events (text, role) tuples. AgentCore
# stores them short-term immediately and queues them for async long-term extraction.
if MEMORY_READY:
    mem.create_event(
        memory_id=memory_id,
        actor_id=ACTOR_ID,
        session_id=SESSION_ID,
        messages=[
            ("Hi, I'm Alex. My monitor from order #A-12345 has a dead pixel.", "USER"),
            ("Sorry about that, Alex — it's within the 30-day window, so you can get a replacement.", "ASSISTANT"),
            ("Yes please, same model. I'm on the Pro plan, by the way.", "USER"),
            ("Done — a Pro-plan replacement for #A-12345 is queued to your address on file.", "ASSISTANT"),
            ("Great. I usually prefer email updates over phone calls.", "USER"),
            ("Noted — I'll send tracking by email once it ships.", "ASSISTANT"),
        ],
    )
    print("Events written for", ACTOR_ID)
else:
    print("Memory not provisioned — skipped.")

In [ ]:
# Short-term recall: exact recent turns of THIS session (no extraction needed).
if MEMORY_READY:
    turns = mem.get_last_k_turns(
        memory_id=memory_id,
        actor_id=ACTOR_ID,
        session_id=SESSION_ID,
        k=5,
    )
    for turn in turns:
        for msg in turn:
            print(msg["role"], "->", msg["content"]["text"])
else:
    print("Memory not provisioned — skipped.")

<div class="alert alert-block alert-warning">
<b>Long-term extraction is asynchronous.</b> Short-term recall above is instant. Long-term facts, though, are distilled by an async LLM pass after <code>create_event</code> — so a retrieve right after writing may return nothing. The next cell waits ~60s, then has a <b>Strands agent</b> retrieve via the AgentCore Memory tool in a fresh session. If it still comes back empty, extraction just hasn't finished — re-run the cell; in production you read whatever facts already exist and let the store catch up in the background.
</div>

In [ ]:
# Long-term retrieve — in a FRESH session — via the Strands-native AgentCore Memory tool.
# AgentCoreMemoryToolProvider exposes record/retrieve over an EXISTING memory; we give a
# Strands agent the retrieve tool and a NEW session_id (Alex "returning tomorrow").
if MEMORY_READY:
    import time
    from strands import Agent
    from strands.models import BedrockModel
    from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

    time.sleep(60)  # give async extraction time to distill facts into the namespace

    provider = AgentCoreMemoryToolProvider(
        memory_id=memory_id,
        actor_id=ACTOR_ID,                       # same user...
        session_id=f"returning-{int(time.time())}",  # ...brand-new session
        namespace=f"/users/{ACTOR_ID}/facts",
        region=REGION,
    )
    returning_agent = Agent(
        model=BedrockModel(model_id=HAIKU, region_name=REGION),
        tools=provider.tools,
        system_prompt=(
            "A returning user just connected. BEFORE replying, use the agent_core_memory "
            "tool to retrieve what you already know about them, then greet them with it."
        ),
    )
    result = returning_agent("Hi, it's Alex again.")
    print(result.message["content"][0]["text"])
else:
    print("Memory not provisioned — skipped.")

In [ ]:
# Cleanup: delete the memory resource so it stops accruing.
if MEMORY_READY:
    mem.delete_memory(memory_id=memory_id)
    print("Deleted:", memory_id)
else:
    print("Memory not provisioned — nothing to clean up.")

**What you'll see**: short-term recall returns the six turns verbatim immediately. Then, in a brand-new session, the Strands agent calls the `agent_core_memory` tool and greets Alex with what it retrieved — the Pro plan, the email preference, the open #A-12345 replacement — *without* replaying the transcript. That's the cross-session win: a handful of extracted facts instead of the whole history. (If the greeting says it has nothing yet, extraction is still running — re-run the cell.)

This is the SDK-native path: `SummarizingConversationManager` bounds cost *within* a session (client-side), and `AgentCoreMemoryToolProvider` carries facts *across* sessions (managed) — you wire both into a Strands `Agent` rather than hand-rolling either.

---

## 6. Batch Inference — Lever 12

If a workload doesn't need a synchronous answer, you can pay half. Bedrock **batch inference** runs requests asynchronously at **50% of the on-demand token price**, SLA ≤24h (small jobs often finish in minutes). It's file-based: build a JSONL of requests, upload to S3, create a *model invocation job*, collect results from S3. The highest-ROI lever for offline work — and the most underused, because teams default to synchronous calls.

| Workload | Batch fit |
|---|---|
| Backfilling embeddings for an existing corpus | ✅ Perfect |
| Nightly summarization of closed tickets | ✅ Perfect |
| Eval runs against a large test set | ✅ Perfect |
| Bulk classification on a CSV export | ✅ Perfect |
| Real-time chat / tool-calling agents | ❌ No — sync only |

<div class="alert alert-block alert-warning">
<b>What batch does NOT support:</b> no tool calling, no structured output, no multi-turn — each record is processed independently — and it's on-demand only (no provisioned). If your workflow needs any of those, batch is the wrong path. Also: <b>match results by <code>recordId</code>, not line order</b> (ordering isn't guaranteed), and a single bad record fails only itself (failures land in a separate error output — inspect both). Bedrock requires a <b>100-record minimum</b> per job; the cell repeats a small ticket set to 120 to clear it.
</div>

In [ ]:
import json as _json
from pathlib import Path

# Bedrock batch requires a MINIMUM of 100 records per job. We repeat a small set to clear it.
BASE_TICKETS = [
    "Customer reports monitor flickers under load. Resolved with firmware update. Closed.",
    "Customer received wrong color phone. Replacement shipped. Closed.",
    "Customer asked about return policy on opened headphones. Policy explained, no return needed. Closed.",
    "Customer's order delayed by carrier. Refund issued for shipping. Closed.",
    "Customer requested warranty info on a refurbished tablet. Provided. Closed.",
]
TICKETS = (BASE_TICKETS * 24)[:120]  # 120 records — comfortably over the 100 minimum

records = [
    {
        "recordId": f"ticket-{i:06d}",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 100,
            "messages": [{
                "role": "user",
                "content": f"Summarize this resolved support ticket in one sentence:\n\n{ticket}",
            }],
        },
    }
    for i, ticket in enumerate(TICKETS)
]

local_path = Path("/tmp/batch-input.jsonl")
with local_path.open("w") as f:
    for r in records:
        f.write(_json.dumps(r) + "\n")

print(f"Wrote {local_path} — {len(records)} records, {local_path.stat().st_size} bytes")
print("\nFirst record:")
print(_json.dumps(records[0], indent=2))

In [ ]:
import os, time, boto3

# Set these to run the job live. Leave BATCH_BUCKET unset to skip submission.
BATCH_BUCKET = os.environ.get("BATCH_BUCKET", "")          # an S3 bucket you can write to
BATCH_ROLE_ARN = os.environ.get("BATCH_ROLE_ARN", "")      # IAM role Bedrock assumes for S3 access

if BATCH_BUCKET and BATCH_ROLE_ARN:
    s3 = boto3.client("s3", region_name=REGION)
    bedrock = boto3.client("bedrock", region_name=REGION)
    prefix = f"batch-demo/{int(time.time())}"

    # 1) Upload the input JSONL to S3
    s3.upload_file(str(local_path), BATCH_BUCKET, f"{prefix}/input.jsonl")
    print(f"Uploaded -> s3://{BATCH_BUCKET}/{prefix}/input.jsonl")

    # 2) Submit the batch job (50% of on-demand price)
    job = bedrock.create_model_invocation_job(
        jobName=f"summaries-{int(time.time())}",
        roleArn=BATCH_ROLE_ARN,
        modelId="anthropic.claude-haiku-4-5-20251001-v1:0",
        inputDataConfig={"s3InputDataConfig": {"s3Uri": f"s3://{BATCH_BUCKET}/{prefix}/input.jsonl"}},
        outputDataConfig={"s3OutputDataConfig": {"s3Uri": f"s3://{BATCH_BUCKET}/{prefix}/output/"}},
    )
    job_arn = job["jobArn"]
    print("Submitted job:", job_arn)

    # 3) Poll until done (small jobs usually finish in a few minutes; SLA is up to 24h)
    while True:
        status = bedrock.get_model_invocation_job(jobIdentifier=job_arn)["status"]
        print("  status:", status)
        if status in ("Completed", "Failed", "Stopped", "Expired"):
            break
        time.sleep(30)

    # 4) Read a few results back from S3
    if status == "Completed":
        out = s3.list_objects_v2(Bucket=BATCH_BUCKET, Prefix=f"{prefix}/output/")
        key = next(o["Key"] for o in out["Contents"] if o["Key"].endswith(".jsonl.out"))
        body = s3.get_object(Bucket=BATCH_BUCKET, Key=key)["Body"].read().decode()
        for line in body.splitlines()[:3]:
            rec = _json.loads(line)
            text = rec["modelOutput"]["content"][0]["text"]
            print(f"  {rec['recordId']}: {text[:90]}")
else:
    print("Set BATCH_BUCKET and BATCH_ROLE_ARN env vars to run the batch job live.")
    print("Flow: upload input.jsonl -> create_model_invocation_job -> poll status -> read s3 output.")
    print("Batch runs at 50% of on-demand price; SLA up to 24h (small jobs finish in minutes).")

**What you'll see**: the first cell writes a 120-record JSONL (over the 100 minimum) and prints one record so you can see the `recordId` + `modelInput` shape. The second submits the job, polls `Submitted -> InProgress -> Completed`, and reads a few summaries back from S3 — all at 50% of the sync price. Without `BATCH_BUCKET`/`BATCH_ROLE_ARN` set, it prints the flow instead of running live.

---

## What you've done

You've worked through the six MEDIUM-effort levers, each a standalone architectural move:

1. **LLM routing** — classify, then send each request to the cheapest model that handles it
2. **Guardrails** — screen traffic at the input layer (inline + standalone `ApplyGuardrail`) so refused requests never cost generation
3. **RAG / indexing** — retrieve only the top-K chunks the query needs
4. **Prompt compression** — strip low-signal tokens from the dynamic tail (never the cached prefix)
5. **Conversation & memory** — bound per-turn context with a sliding window + rolling summary, persist across sessions with AgentCore Memory
6. **Batch inference** — run offline work asynchronously at half price

These layer on top of the LOW-effort levers — apply them where they fit, measure on your own eval set, and reach for the HIGH-effort tier only once these are exhausted.

**Next**: [`03-high-effort.ipynb`](./03-high-effort.ipynb) for the compound-AI patterns.